In [1]:
!pip install -q transformers accelerate sentencepiece

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [3]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [4]:
def generate(prompt):

    messages = [
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7
    )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    print(response)

Zero-Shot Prompting

In [5]:
prompt = """
Translate the following sentence into French.

Sentence:
Machine learning is changing the world.
"""

generate(prompt)

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user

Translate the following sentence into French.

Sentence:
Machine learning is changing the world.

assistant
La machine learning change le monde.


In [6]:
prompt = """
Summarize the following paragraph.

Artificial Intelligence is transforming healthcare by assisting doctors in disease diagnosis, drug discovery, and patient monitoring.
"""

generate(prompt)

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user

Summarize the following paragraph.

Artificial Intelligence is transforming healthcare by assisting doctors in disease diagnosis, drug discovery, and patient monitoring.

assistant
Artificial intelligence (AI) is revolutionizing healthcare through several key areas:

1. Disease Diagnosis: AI can analyze vast amounts of medical data to identify patterns that might not be apparent to human clinicians. This includes using machine learning algorithms to predict diseases from symptoms, track the progression of conditions over time, and even suggest personalized treatment plans based on individual genetic profiles.

2. Drug Discovery: AI accelerates the process of discovering new drugs by automating complex chemical analysis, predicting compound efficacy, and optimizing drug formulation processes. It helps researchers design more effective medications with improved safety and efficacy.

3. Patient Monitoring: A

One-Shot Prompting

In [7]:
prompt = """
Example:

English: Good Morning
French: Bonjour

Now translate.

English: How are you?
French:
"""

generate(prompt)

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user

Example:

English: Good Morning
French: Bonjour

Now translate.

English: How are you?
French:

assistant
Bonjour, comment ça va?


In [8]:
prompt = """
Example

Question:
What is AI?

Answer:
Artificial Intelligence enables computers to perform tasks that normally require human intelligence.

Question:
What is Machine Learning?

Answer:
"""

generate(prompt)

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user

Example

Question:
What is AI?

Answer:
Artificial Intelligence enables computers to perform tasks that normally require human intelligence.

Question:
What is Machine Learning?

Answer:

assistant
Machine Learning is a subset of artificial intelligence (AI) that involves the development of algorithms and statistical models that allow machines to learn from data and improve their performance over time without being explicitly programmed. This process typically involves training the machine on examples or patterns in order to identify patterns and make predictions about new data points. Machine learning techniques can be applied to a wide range of applications, including image recognition, speech recognition, natural language processing, and predictive analytics.


Few-Shot Prompting

In [9]:
prompt = """
Example 1

Positive Review:
The movie was fantastic.

Sentiment:
Positive

Example 2

Review:
The food was terrible.

Sentiment:
Negative

Example 3

Review:
The service was excellent.

Sentiment:
Positive

Now classify.

Review:
The laptop is slow and keeps crashing.

Sentiment:
"""

generate(prompt)

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user

Example 1

Positive Review:
The movie was fantastic.

Sentiment:
Positive

Example 2

Review:
The food was terrible.

Sentiment:
Negative

Example 3

Review:
The service was excellent.

Sentiment:
Positive

Now classify.

Review:
The laptop is slow and keeps crashing.

Sentiment:

assistant
Negative


In [10]:
prompt = """
Example

2 + 3 = 5

10 + 15 = 25

7 + 12 = 19

Now solve

34 + 19 =
"""

generate(prompt)

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user

Example

2 + 3 = 5

10 + 15 = 25

7 + 12 = 19

Now solve

34 + 19 =

assistant
Sure! Let's solve the problem step-by-step:

1) First, we need to add the numbers in the first part: 34 and 19.
   \[
   34 + 19 = 53
   \]

2) Now, we take the result from the first addition (53), add it to the number in the second part (7).
   \[
   53 + 7 = 60
   \]

So, the final answer is:
\[
\boxed{60}
\]


Chain-of-Thought Prompting

In [11]:
prompt = """
A shopkeeper bought a pen for $10 and sold it for $15.

Explain your reasoning step by step and calculate the profit.
"""

generate(prompt)

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user

A shopkeeper bought a pen for $10 and sold it for $15.

Explain your reasoning step by step and calculate the profit.

assistant
Certainly! Let's break down the problem step by step to determine the profit:

### Step 1: Calculate the Cost Price (CP)
The cost price is the price at which the item was purchased. In this case:
- The pen was bought for $10.
- Therefore, the cost price is $10.

### Step 2: Calculate the Selling Price (SP)
The selling price is the amount earned from selling the item. In this case:
- The seller sold the pen for $15.
- Thus, the selling price is $15.

### Step 3: Determine the Profit
Profit is calculated as the difference between the selling price and the cost price. We can use the formula:
\[ \text{Profit} = \text{Selling Price} - \text{Cost Price} \]

Substituting the values we have:
\[ \text{Profit} = \$15 - \$10 = \$5 \]

So, the profit


In [12]:
prompt = """
John has 12 apples.

He gives 5 apples to Mary.

Mary returns 2 apples.

Think step by step before answering.

How many apples does John have now?
"""

generate(prompt)

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user

John has 12 apples.

He gives 5 apples to Mary.

Mary returns 2 apples.

Think step by step before answering.

How many apples does John have now?

assistant
Let's think through this step-by-step:

1. Initially, John has 12 apples.
2. He gives away 5 apples: \( 12 - 5 = 7 \) apples left with him.
3. Mary then returns 2 apples: \( 7 + 2 = 9 \) apples.

So, after all these transactions, John has 9 apples left.


Compare Outputs

In [13]:
prompts = {

"Zero Shot":
"Who invented the telephone?",

"One Shot":
"""
Question:
Capital of India?

Answer:
New Delhi

Question:
Capital of France?

Answer:
""",

"Few Shot":
"""
Dog -> Animal

Rose -> Flower

Apple -> Fruit

Carrot ->
""",

"Chain of Thought":
"""
A train travels 60 km in one hour.

How far will it travel in 5 hours?

Think step by step.
"""
}

for name, prompt in prompts.items():

    print("="*60)
    print(name)
    print("="*60)

    generate(prompt)

    print("\n")

Zero Shot
system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Who invented the telephone?
assistant
The invention of the telephone is attributed to Alexander Graham Bell and his father, Thomas Edison. The two men worked together in the late 19th century, creating one of the most significant technological advancements of the 20th century.

Alexander Graham Bell was born on May 6, 1847, in Dublin, Ireland, while his father, Thomas Edison, also had an influence on the development of the telephone. Both Bell and Edison were involved in the creation of the phonograph and later the phonograph needle, which played a crucial role in the evolution of communication technology.

Bell's work with Thomas Edison led to the development of the first practical telegraph machine, which was used for transmitting messages over long distances. However, it wasn't until after Edison died that Bell began to focus solely on the telephone.

In 1876, Bell demonstrated the first succe